# GI Lab: Weisfeiler–Leman experiments

In Colab, choose **Runtime → Run all** from a fresh runtime. The first code cell downloads the repository, installs plotting support, and imports `Graph` and the algorithms before the examples run. Do not start by running an example cell on its own.

This workbench compares 1-WL and 2-WL graph-isomorphism tests, explores examples, and checks small graphs with an exact oracle.

In [ ]:
# Find a local checkout or download the public repository (Colab starts without the package installed).
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "gi_lab").is_dir():
    ROOT = ROOT / "gi-lab-repository"
    if not (ROOT / "gi_lab").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/shrnasa011/testing_isomorphism.git", str(ROOT)],
            check=True,
        )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%pip install -q matplotlib

from gi_lab.algorithms import ColorRefinement, TwoWL
from gi_lab.exact import is_isomorphic
from gi_lab.families import (
    cycle, disjoint_cycles, rook_graph, shrikhande_graph,
    default_corpus,
)
from gi_lab.experiments import run
from gi_lab.graph import Graph
from gi_lab.search import find_collision
from gi_lab.draw import draw_pair


## Build graphs by hand

`Graph.from_edges` is the most direct way to encode a small example. Isolated vertices can be supplied explicitly.

In [ ]:
triangle_plus_isolate = Graph.from_edges(
    [(0, 1), (1, 2), (2, 0)],
    vertices=range(4),
)
triangle_plus_isolate.describe(), triangle_plus_isolate.edge_list()

## Compare a known 1-WL failure

The 12-cycle and two disjoint 6-cycles are not isomorphic, but every vertex has degree two. Therefore 1-WL reaches the same single-colour partition. 2-WL distinguishes them.

In [ ]:
left, right = cycle(12), disjoint_cycles(6, 6)
for algorithm in (ColorRefinement(), TwoWL()):
    same = algorithm.refine(left).signature == algorithm.refine(right).signature
    print(f'{algorithm.name}: same signature = {same}')
print('exactly isomorphic:', is_isomorphic(left, right))
draw_pair(left, right, ('Random graph A', 'Random graph B'))

## Run the curated regression corpus

A `collision` means a non-isomorphic pair that this invariant cannot separate; it is not a bug in the runner.

In [ ]:
trials = run(default_corpus(include_permutations=True), [ColorRefinement(), TwoWL()])
for trial in trials:
    print(f'{trial.case:26} {trial.algorithm:3}  {trial.verdict}')

## Search randomly for a test-breaking pair

Random regular graphs are a productive source of 1-WL collisions because all vertices begin with the same degree. The seed makes the search repeatable. The exact oracle validates any reported collision, and the edge lists let you turn it into a permanent hand-built case.

Pass `connected_only=True` to `find_collision` to skip disconnected samples and search for collisions among connected graphs only.

For Erdős–Rényi samples, use `model="erdos"` and either `average_degree=d` (which sets `p = d / (vertices - 1)`) or `probability=p`. These options are mutually exclusive.

In [ ]:
collision = find_collision(
    ColorRefinement(), vertices=10, degree=2, samples=40, seed=7
)
if collision is None:
    print('No collision found within this budget; try more samples or a new seed.')
else:
    print(collision.reproduce())
    draw_pair(collision.left, collision.right, ('Random graph A', 'Random graph B'))

### Search Erdős–Rényi graphs by expected average degree

Here `average_degree=4` sets the edge probability to `4 / (10 - 1)`. `connected_only=True` skips disconnected samples.

In [ ]:
erdos_collision = find_collision(
    ColorRefinement(), vertices=5, model="erdos",
    average_degree=3, samples=400, seed=7, connected_only=True
)
if erdos_collision is None:
    print("No collision found within this sample budget.")
else:
    print(erdos_collision.reproduce())
    draw_pair(erdos_collision.left, erdos_collision.right,
              ("Erdos graph A", "Erdos graph B"))

## A stronger regular-graph example

The rook and Shrikhande graphs are non-isomorphic strongly regular graphs with the same parameters. They collide under both included WL variants.

In [ ]:
rook, shrikhande = rook_graph(), shrikhande_graph()
print('exactly isomorphic:', is_isomorphic(rook, shrikhande))
for algorithm in (ColorRefinement(), TwoWL()):
    print(algorithm.name, algorithm.refine(rook).signature == algorithm.refine(shrikhande).signature)
draw_pair(rook, shrikhande, ('4×4 rook graph', 'Shrikhande graph'))

## Use edge lists from the homepage playground

Paste the same one-edge-per-line format used by the homepage graph playground. Labels can be numbers or words. This section runs 1-WL, 2-WL, and the exact checker from this workbench, then draws the pair. The exact check is intended for small examples.


In [ ]:
def graph_from_edge_text(text):
    edges = []
    vertices = set()
    for line_number, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.replace(",", " ").split()
        if len(parts) != 2 or parts[0] == parts[1]:
            raise ValueError(f"Line {line_number}: enter two different vertex labels.")
        left, right = parts
        edges.append((left, right))
        vertices.update((left, right))
    if not edges:
        raise ValueError("Enter at least one edge.")
    return Graph.from_edges(edges, vertices=vertices)

# C6 versus two disjoint triangles: same 1-WL signature, not isomorphic.
edge_text_a = """0 1
1 2
2 3
3 4
4 5
5 0"""
edge_text_b = """0 1
1 2
2 0
3 4
4 5
5 3"""

graph_a = graph_from_edge_text(edge_text_a)
graph_b = graph_from_edge_text(edge_text_b)
for algorithm in (ColorRefinement(), TwoWL()):
    same = algorithm.refine(graph_a).signature == algorithm.refine(graph_b).signature
    print(f"{algorithm.name}: signatures match = {same}")
print("exactly isomorphic:", is_isomorphic(graph_a, graph_b))
draw_pair(graph_a, graph_b, ("Graph A", "Graph B"))


## Animate the same pair with Graph Animation

The [Graph Animation project](https://github.com/shrnasa011/graph_animation) uses Manim. In Colab, run the install cell, restart the runtime when it finishes, then run the load and scene cells. Locally, install Manim and its system dependencies using the project README before running the load and scene cells.


In [ ]:
# Install Manim for Colab. Locally, use the Graph Animation README for system dependencies.
import subprocess
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "libcairo2-dev", "libpango1.0-dev", "ffmpeg"], check=True)
%pip install -q manim==0.20.1 ipython==8.21.0
print("If running in Colab, restart the runtime after installation, then run the load and scene cells.")


In [ ]:
from pathlib import Path
import subprocess
import sys

ANIMATION_ROOT = Path.cwd().resolve().parent / "Graph Animation"
if not (ANIMATION_ROOT / "graph_animation").is_dir():
    ANIMATION_ROOT = Path.cwd().resolve() / "graph-animation-repository"
    if not (ANIMATION_ROOT / "graph_animation").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shrnasa011/graph_animation.git", str(ANIMATION_ROOT)], check=True)
if str(ANIMATION_ROOT) not in sys.path:
    sys.path.insert(0, str(ANIMATION_ROOT))

%load_ext manim
from graph_animation.graph import Graph as AnimatedGraph


In [ ]:
%%manim -v WARNING -ql PlaygroundGraphPair
from manim import *
import numpy as np

class PlaygroundGraphPair(Scene):
    def construct(self):
        self.camera.background_color = WHITE
        def positions_for(vertices, center_x):
            angles = np.linspace(0, 2 * np.pi, len(vertices), endpoint=False)
            return {v: np.array([center_x + 1.35*np.cos(a), 1.15*np.sin(a), 0]) for v, a in zip(vertices, angles)}

        left_vertices = [str(v) for v in graph_a.vertices]
        right_vertices = [str(v) for v in graph_b.vertices]
        left = AnimatedGraph(vertices=left_vertices, edges=[tuple(map(str, edge)) for edge in graph_a.edge_list()], positions=positions_for(left_vertices, -3.2))
        right = AnimatedGraph(vertices=right_vertices, edges=[tuple(map(str, edge)) for edge in graph_b.edge_list()], positions=positions_for(right_vertices, 3.2))
        title = Text("1-WL collision: exact check says not isomorphic", font_size=25)
        title.to_edge(UP)
        self.add(title, left.view, right.view)
        self.play(left.show_degrees(), right.show_degrees())
        self.play(left.show_halo("0"), right.show_halo("0"))
        self.wait(1)
